# Remedia — Modal Uyumlu Giriş Notebook'u

Önerilen kullanım: **remedia-modal / notebook_image** hazır image'ını seç ve bu notebook'u çalıştır.

Hazır image algılanırsa paket kurulumu tamamen atlanır. Standart Modal image kullanılırsa gerekli CUDA kütüphaneleri yedek olarak otomatik kurulur.


In [ ]:
import os
import site
import subprocess
import sys
from pathlib import Path

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("GPU görünmüyor. Compute ekranından L4 seçip kernel'i yeniden başlat.")

prebuilt = os.environ.get("REMEDIA_PREBUILT_IMAGE") == "1"
cuda_packages = [
    "nvidia-cuda-runtime-cu12>=12.4",
    "nvidia-cublas-cu12>=12.4",
    "nvidia-cusparse-cu12>=12.3",
    "nvidia-cusolver-cu12>=11.6",
    "nvidia-curand-cu12>=10.3",
    "nvidia-nvjitlink-cu12>=12.4",
    "nvidia-nvtx-cu11==11.8.86",
]
if prebuilt:
    print("⚡ Hazır Remedia image algılandı; paket kurulumu atlandı.")
else:
    print("📦 Standart image kullanılıyor; GNINA runtime bir kez hazırlanıyor...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *cuda_packages],
        check=True,
    )

library_dirs = []
prelinked = Path("/opt/remedia-nvidia-libs")
if prelinked.is_dir():
    library_dirs.append(str(prelinked))
for base in [*site.getsitepackages(), site.getusersitepackages()]:
    nvidia_root = Path(base) / "nvidia"
    if nvidia_root.exists():
        library_dirs.extend(str(path) for path in nvidia_root.glob("*/lib") if path.is_dir())

existing = [p for p in os.environ.get("LD_LIBRARY_PATH", "").split(":") if p]
merged = list(dict.fromkeys([*library_dirs, *existing]))
os.environ["LD_LIBRARY_PATH"] = ":".join(merged)

required_libraries = {
    "libcusparse.so.12": "nvidia-cusparse-cu12",
    "libnvToolsExt.so.1": "nvidia-nvtx-cu11==11.8.86",
}
for library, package in required_libraries.items():
    matches = [Path(path) / library for path in library_dirs if (Path(path) / library).exists()]
    if not matches:
        raise RuntimeError(f"{library} bulunamadı ({package}); GNINA başlatılamaz.")

gnina = Path(os.environ.get("GNINA_PATH", "/usr/local/bin/gnina"))
if gnina.exists():
    check = subprocess.run([str(gnina), "--version"], capture_output=True, text=True, env=os.environ)
    if check.returncode != 0:
        raise RuntimeError(check.stderr.strip() or "GNINA başlatılamadı.")

print("✅ GPU ve GNINA runtime hazır")


In [ ]:
import json
import shutil
import time
import urllib.request
from pathlib import Path

for stale_repo in (Path('/mnt/remedia-data/Remedia'), Path('/root/remedia_workspace/Remedia')):
    if stale_repo.exists():
        shutil.rmtree(stale_repo)
        print(f'♻️ Eski kod cache’i temizlendi: {stale_repo}')

FORM_NOTEBOOK_URL = (
    "https://raw.githubusercontent.com/mehmetg06/Remedia/"
    "main/notebooks/remedia_modal.ipynb"
)
request = urllib.request.Request(
    f"{FORM_NOTEBOOK_URL}?t={time.time_ns()}",
    headers={"Cache-Control": "no-cache", "User-Agent": "Remedia-Modal"},
)
with urllib.request.urlopen(request, timeout=60) as response:
    source_notebook = json.load(response)

code_cells = [
    cell.get("source", "")
    for cell in source_notebook.get("cells", [])
    if cell.get("cell_type") == "code"
]
if not code_cells:
    raise RuntimeError("Remedia form kodu bulunamadı.")

def normalize_source(source):
    return "".join(source) if isinstance(source, list) else source

form_source = "\n\n".join(normalize_source(cell) for cell in code_cells)
exec(compile(form_source, FORM_NOTEBOOK_URL, "exec"), globals())
